# Part 1: Pre-Filtering vs. Post-Filtering Patterns

When building a robust RAG pipeline, semantic search alone often falls short. Users look for information tied to strict context boundaries—such as file permissions, temporal limits ("documents published after 2024"), or categorization ("only look in the HR department").

To solve this, we inject metadata (key-value pairs attached to text chunks during document parsing) and enforce rules using two primary design patterns: Pre-Filtering and Post-Filtering.

## 1. Pre-Filtering (Filter-First Architecture)
### How It Works Under the Hood
In a pre-filtering pattern, the vector database evaluates the metadata conditions before running any vector similarity calculations.

The database checks its metadata index (inverted index or boolean bitmask) to find document chunk IDs matching your criteria (e.g., department == "finance").

It completely discards non-matching chunks from the search pool.

The Approximate Nearest Neighbor (ANN) search algorithm (like HNSW or IVF) computes vector distances strictly within the filtered subset.

### Code Implementation Pattern
Here is how pre-filtering is typically executed using a vector store (e.g., Chroma or Pinecone) via LangChain:

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Initialize embeddings and vector store
embeddings = OpenAIEmbeddings()
vector_store = Chroma(
    collection_name="enterprise_docs",
    embedding_function=embeddings
)

# Define a strict pre-filter condition
# This restricts the search space before vector math is computed
finance_filter = {
    "$and": [
        {"department": {"$eq": "finance"}},
        {"year": {"$gt": 2024}}
    ]
}

# Execute search with pre-filtering
results = vector_store.similarity_search(
    query="What is our projected Q3 budget?",
    k=4,
    filter=finance_filter  # Evaluated by the vector DB engine first
)

### Trade-offs & Production Risks

**Pros:**
Absolute Precision: Zero risk of returning a document that violates your metadata boundaries. Perfect for multi-tenant isolation (e.g., tenant_id == "company_a").
Compute Efficiency: Saves processing cycles because similarity math is performed on a smaller slice of data.

**Cons:**
(The Recall Starvation Risk): If your filter is too specific (e.g., filtering down to a category that only has 2 chunks total), but you requested $k=5$, the database can only return those 2 chunks. Furthermore, in graph-based vector indexes (like HNSW), heavy pre-filtering can isolate graph nodes, degrading search recall.

## 2. Post-Filtering (Similarity-First Architecture)
**How It Works Under the HoodIn a post-filtering pattern, the sequence is inverted:**

The vector database ignores metadata initially and executes a global vector similarity search across the entire collection to fetch a large candidate pool (e.g., top $50$ matches).
Your application code (or a post-processing layer) iterates through those results and drops any chunks that fail to satisfy the metadata constraints.
Finally, the list is sliced down to your target size ($Top\text{-}K$, e.g., top $5$).

**code implementation pattern**

In [ ]:
# Step 1: Perform a broad semantic search to gather a large candidate pool
raw_results = vector_store.similarity_search(
    query="What is the process for onboarding new employees?",
    k=50  # Fetching a larger pool to compensate for downstream drops
)

# Step 2: Apply manual post-hoc metadata filtering in application memory
filtered_results = [
    doc for doc in raw_results 
    if doc.metadata.get("department") == "HR" and doc.metadata.get("status") == "active"
][:5]  # Slice down to final desired Top-K

Trade-offs & Production Risks
Pros:

Preserves Semantic Scope: The initial search relies purely on semantic meaning across the whole corpus, ensuring you don't prematurely block relevant context.  

Cons (The Shortfall Problem): 

If your post-filter is restrictive, you might fetch 50 items, but discover that only 1 of them belongs to the target metadata category. Your user ends up with a single context chunk (or zero chunks), even though hundreds of relevant documents exist globally. It also wastes compute embedding and scoring vectors that get thrown away.

## Summary Cheat Sheet for Your Study Notes

| Dimension | Pre-Filtering | Post-Filtering
| :--- | :--- | :--- |
| Execution Sequence | Filter → Vector Search | Vector Search → Filter
| Primary Strength | "Guaranteed safety, multi-tenant security compliance" | Broad semantic exploration before refinement
| Primary Failure Mode | Recall starvation (returning fewer than k chunks) | Empty/shortfall results or wasted search compute
| Best Use Case | "Hard boundaries (Tenant ID, Security Roles, Date cutoffs)" | Soft preferences or optional secondary refinements